# 05 - Train Bed Model (Memory Optimized)

Train with ~12K balanced samples (float32) - very light on memory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/smart-ambulance-ml'
!pip install -q xgboost lightgbm joblib matplotlib

import numpy as np
import pandas as pd
import gc
import matplotlib.pyplot as plt
import joblib
import json
import time
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Load data
print("Loading bed data...")
data = np.load(f'{PROJECT_DIR}/datasets/processed/bed_train.npz', allow_pickle=True)
X_train = data['X_train'].astype(np.float32)
X_val = data['X_val'].astype(np.float32)
X_test = data['X_test'].astype(np.float32)
y_train = data['y_train'].astype(np.float32)
y_val = data['y_val'].astype(np.float32)
y_test = data['y_test'].astype(np.float32)
feature_names = data['feature_names'].tolist()

print(f"✓ Loaded: Train {X_train.shape}, Val {X_val.shape}, Test {X_test.shape}")
print(f"RAM usage: ~{X_train.nbytes / 1e6:.1f}MB for training data")

In [ ]:
print("\n="*60)
print("TRAINING BED MODELS")
print("="*60)

models = {
    'Random Forest': RandomForestRegressor(
        n_estimators=150, max_depth=16, min_samples_split=4,
        random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBRegressor(
        n_estimators=150, max_depth=6, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85,
        random_state=42, n_jobs=-1,
        tree_method='hist'
    ),
    'LightGBM': LGBMRegressor(
        n_estimators=150, learning_rate=0.05, num_leaves=31,
        subsample=0.85, colsample_bytree=0.85,
        random_state=42, n_jobs=-1, verbose=-1
    )
}

results = {}
trained_models = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    start = time.time()
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    y_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'time': time.time() - start}
    print(f"  ✓ MAE: {mae:.3f} beds, RMSE: {rmse:.3f}, R2: {r2:.4f}")
    gc.collect()

print("\n" + pd.DataFrame(results).T.round(4).to_string())

In [ ]:
print("\n="*60)
print("TRAINING STACKING REGRESSOR")
print("="*60)

base_estimators = [(name, model) for name, model in trained_models.items()]
meta_learner = Ridge(alpha=1.0)

stacking_model = StackingRegressor(
    estimators=base_estimators,
    final_estimator=meta_learner,
    cv=3,
    n_jobs=-1
)

start = time.time()
stacking_model.fit(X_train, y_train)

y_pred = stacking_model.predict(X_val)
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
r2 = r2_score(y_val, y_pred)

results['Stacking'] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'time': time.time() - start}
trained_models['Stacking'] = stacking_model

print(f"✓ Stacking: MAE={mae:.3f} beds, RMSE={rmse:.3f}, R2={r2:.4f}")
gc.collect()

In [ ]:
print("\n="*60)
print("TEST EVALUATION")
print("="*60)

test_results = {}
for name, model in trained_models.items():
    y_pred = np.round(model.predict(X_test)).astype(int)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    test_results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}

test_df = pd.DataFrame(test_results).T.sort_values('MAE')
print(test_df.round(4))

print(f"\n✓ Best: {test_df.index[0]} (MAE={test_df.iloc[0]['MAE']:.2f} beds)")

In [ ]:
# Visualization
sample_idx = slice(0, min(200, len(y_test)))
y_sample = y_test[sample_idx]
y_pred_sample = trained_models[test_df.index[0]].predict(X_test[sample_idx])

plt.figure(figsize=(10, 5))
plt.plot(y_sample, label='Actual', marker='o', markersize=3, alpha=0.7)
plt.plot(y_pred_sample, label='Predicted', marker='x', markersize=3, alpha=0.7)
plt.xlabel('Sample Index')
plt.ylabel('Available Beds')
plt.title(f'Bed Availability: Actual vs Predicted\nBest model: {test_df.index[0]}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/bed_predictions.png', dpi=150)
plt.show()

In [ ]:
print("\n="*60)
print("SAVING MODELS")
print("="*60)

models_dir = f'{PROJECT_DIR}/models/bed_availability'
os.makedirs(models_dir, exist_ok=True)

for name, model in trained_models.items():
    filename = name.lower().replace(' ', '_') + '.joblib'
    joblib.dump(model, f'{models_dir}/{filename}')
    print(f"✓ Saved {name}")

# Update registry
try:
    with open(f'{PROJECT_DIR}/models/model_registry.json', 'r') as f:
        registry = json.load(f)
except:
    registry = {}

registry['bed'] = {
    'models': list(trained_models.keys()),
    'best_model': test_df.index[0],
    'test_mae': test_results[test_df.index[0]]['MAE'],
    'test_r2': test_results[test_df.index[0]]['R2']
}

with open(f'{PROJECT_DIR}/models/model_registry.json', 'w') as f:
    json.dump(registry, f, indent=2)

print(f"\n✓ Bed training complete!")
gc.collect()